# Orpheus TTS — English production voice test
Run in Google Colab. Select a GPU runtime if available. This notebook uses the official Orpheus English finetuned production checkpoint.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install orpheus-speech vllm==0.7.3


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('A CUDA GPU is normally required by the official vLLM inference path. Choose a Colab GPU runtime if available.')


## Generate the shared comparison sample
Orpheus uses a built-in English speaker. Choose one of the available voices: tara, leah, jess, leo, dan, mia, zac, or zoe.


In [ ]:
from orpheus_tts import OrpheusModel
import wave, os
assert torch.cuda.is_available(), 'A CUDA GPU is required for this notebook.'
voice = 'tara'
sample_text = 'Welcome to the voice comparison. This is a natural English speech sample generated with an open source text to speech model. The goal is clear pronunciation, smooth pacing, and a confident, conversational delivery.'
model = OrpheusModel(model_name='canopylabs/orpheus-tts-0.1-finetune-prod', max_model_len=2048)
audio_chunks = model.generate_speech(prompt=sample_text, voice=voice)
wav_path = 'orpheus_english_sample.wav'
with wave.open(wav_path, 'wb') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    for chunk in audio_chunks:
        wf.writeframes(chunk)
print('Saved', wav_path)


In [ ]:
from IPython.display import Audio, display
from google.colab import files
display(Audio('orpheus_english_sample.wav'))
!ffmpeg -y -loglevel error -i orpheus_english_sample.wav -codec:a libmp3lame -qscale:a 3 orpheus_english_sample.mp3
files.download('orpheus_english_sample.mp3')
